# ML - NAIVE BAYES

In [28]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import classification_report
from utils1 import get_classifier_metrics
import re
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Paso 1: Carga del conjunto de datos


In [29]:
df = pd.read_csv('../data/raw/playstore_reviews.csv')
df

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0
...,...,...,...
886,com.rovio.angrybirds,loved it i loooooooooooooovvved it because it...,1
887,com.rovio.angrybirds,all time legendary game the birthday party le...,1
888,com.rovio.angrybirds,ads are way to heavy listen to the bad review...,0
889,com.rovio.angrybirds,fun works perfectly well. ads aren't as annoy...,1


# Paso 2: Estudio de variables y su contenido

In [30]:
df = df.drop(columns=['package_name'])

In [31]:
df

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0
...,...,...
886,loved it i loooooooooooooovvved it because it...,1
887,all time legendary game the birthday party le...,1
888,ads are way to heavy listen to the bad review...,0
889,fun works perfectly well. ads aren't as annoy...,1


In [32]:
# Pasamos la columna review a texto limpio (eliminamos todo lo que no sea letras o espacios, reemplazamos dobles espacios por uno y convertimos a minuscyulas)
df["review"] = df["review"].str.lower().apply(lambda x: re.sub(r'\s+', ' ', re.sub(r'[^a-z\s]', '', x))).str.strip()

x_nb = df['review']
y_nb = df['polarity']

x_train_nb, x_test_nb, y_train_nb, y_test_nb = train_test_split(x_nb, y_nb, test_size=0.2, random_state=21)

In [ ]:
vec_model = CountVectorizer(stop_words = "english")  
x_train_nb = vec_model.fit_transform(x_train_nb).toarray()
x_test_nb = vec_model.transform(x_test_nb).toarray()

In [ ]:
# Visualizamos las palabras que ha extraido CountVectorizer
vocabulary = vec_model.get_feature_names_out()
vocabulary

array(['aakhirat', 'aap', 'aapsssssss', ..., 'zespole', 'zoom', 'zooming'],
      shape=(3330,), dtype=object)

# Paso 3: Construimos los modelos de naive bayes

In [34]:
multinomial_nbm = MultinomialNB()
multinomial_nbm.fit(x_train_nb, y_train_nb)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [35]:
y_pred_test_mn = multinomial_nbm.predict(x_test_nb)
y_pred_train_mn = multinomial_nbm.predict(x_train_nb)

In [36]:
default_mn = get_classifier_metrics(y_pred_test_mn, y_test_nb, y_pred_train_mn, y_train_nb, average='weighted')
default_mn

,Accuracy,F1 Score,Precision,Recall
Train set,0.952247,0.952099,0.952101,0.952247
Test set,0.815642,0.811364,0.813940,0.815642


In [37]:
report_nbm= classification_report(y_test_nb, y_pred_test_mn)
print(report_nbm)

              precision    recall  f1-score   support

           0       0.82      0.90      0.86       114
           1       0.80      0.66      0.72        65

    accuracy                           0.82       179
   macro avg       0.81      0.78      0.79       179
weighted avg       0.81      0.82      0.81       179



In [38]:
bernoulli_nbm = BernoulliNB()
bernoulli_nbm.fit(x_train_nb, y_train_nb)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [39]:
y_pred_test_b = bernoulli_nbm.predict(x_test_nb)
y_pred_train_b = bernoulli_nbm.predict(x_train_nb)

In [40]:
default_bernoulli = get_classifier_metrics(y_pred_test_b, y_test_nb, y_pred_train_b, y_train_nb, average='weighted')
default_bernoulli

,Accuracy,F1 Score,Precision,Recall
Train set,0.922753,0.920191,0.929289,0.922753
Test set,0.776536,0.751464,0.805641,0.776536


In [41]:
report_bernoulli_nb= classification_report(y_test_nb, y_pred_test_b)
print(report_bernoulli_nb)

              precision    recall  f1-score   support

           0       0.75      0.97      0.85       114
           1       0.90      0.43      0.58        65

    accuracy                           0.78       179
   macro avg       0.83      0.70      0.72       179
weighted avg       0.81      0.78      0.75       179



In [42]:
gaussian_nbm = GaussianNB()
gaussian_nbm.fit(x_train_nb, y_train_nb)

,priors,None
,var_smoothing,1e-09


In [43]:
y_pred_test_gauss = gaussian_nbm.predict(x_test_nb)
y_pred_train_gauss = gaussian_nbm.predict(x_train_nb)

In [44]:
default_gaussian = get_classifier_metrics(y_pred_test_gauss, y_test_nb, y_pred_train_gauss, y_train_nb, average='weighted')
default_gaussian

,Accuracy,F1 Score,Precision,Recall
Train set,0.984551,0.984631,0.985222,0.984551
Test set,0.821229,0.820610,0.820206,0.821229


In [45]:
report_gaussian_nb= classification_report(y_test_nb, y_pred_test_gauss)
print(report_gaussian_nb)

              precision    recall  f1-score   support

           0       0.85      0.87      0.86       114
           1       0.76      0.74      0.75        65

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



> ## Conclusiones:
>
> - El modelo por defecto con mejores métricas es el `GaussianNB`, ya que tiene mejor f1-score, métrica que vamos a dar prioridad para nuestro modelo de clasificación de reseñas.
> - Vamos a intentar optimizarlo probando con los hiperparámetros de `GridSearchCV` y `RandomForestClassifier`

# Paso 4: Optimizamos con RandomForestClassifier y GridSearchCV

In [46]:
rf = RandomForestClassifier(random_state=21, class_weight='balanced')

param_grid = {'n_estimators': list(range(70, 90, 5)), 
              'criterion':['gini','entropy'],
              'max_depth' : [5, 7],
              'min_samples_leaf': [3, 4, 5]}

grid = GridSearchCV(rf,
                    param_grid,
                    cv=5,
                    scoring='f1_macro') # Nos interesa las dos clases por igual, tanto reseñas negativas como positivas

In [48]:
# Entrenamos el grid con los hiperparametros
grid.fit(x_train_nb, y_train_nb)
# Devolvemos los mejores parametros despues de entrenarlo
grid.best_params_

{'criterion': 'entropy',
 'max_depth': 7,
 'min_samples_leaf': 4,
 'n_estimators': 80}

In [49]:
# Modelos con los mejores parametros
rf_grid = grid.best_estimator_

In [50]:
# Repetimos el entrenamiento pero ahora con el grid que tiene los hiperparametros establecidos
rf_grid.fit(x_train_nb, y_train_nb)
# Predicciones
y_pred_test_grid = rf_grid.predict(x_test_nb)
y_pred_train_grid = rf_grid.predict(x_train_nb)

In [51]:
grid_rf_metrics = get_classifier_metrics(y_pred_test_grid, y_test_nb, y_pred_train_grid, y_train_nb, average='weighted')
grid_rf_metrics

,Accuracy,F1 Score,Precision,Recall
Train set,0.856742,0.859273,0.869075,0.856742
Test set,0.826816,0.829182,0.837830,0.826816


In [52]:
report_grid_rf= classification_report(y_test_nb, y_pred_test_grid)
print(report_grid_rf)

              precision    recall  f1-score   support

           0       0.90      0.82      0.86       114
           1       0.72      0.85      0.78        65

    accuracy                           0.83       179
   macro avg       0.81      0.83      0.82       179
weighted avg       0.84      0.83      0.83       179



In [53]:
# Creamos un dataframe para comparar las métricas de test de ambos modelos
data = {'Métrica': ['Accuracy', 'F1 Score', 'Precision', 'Recall'],
        'GaussianNB': [0.821, 0.821, 0.820, 0.821],
        'Random Forest': [0.827, 0.829, 0.838, 0.827]}

df_comparacion = pd.DataFrame(data)
df_comparacion

,Métrica,GaussianNB,Random Forest
0,Accuracy,0.821,0.827
1,F1 Score,0.821,0.829
2,Precision,0.820,0.838
3,Recall,0.821,0.827


In [54]:
#Guardamos el mejor modelo
with open('../models/07-rf-grid', 'wb') as f:
    pickle.dump(rf_grid, f)

> ## Conclusiones:
>
> - Hemos mejorado ligeramente las métricas de test
> - El f1-score ha mejorado en la reseñas positivas, ha subido en 3% en la clase 1
> - El modelo es más robusto por lo que generaliza mejor, tiene menos overfitting
